# CNN-EXP-046 — R32 Feasibility Frontier

**Diagnostic-only stage before the frozen regional-head pilot.**

The primary region size remains **pre-registered at R=32 bp**. This notebook must not use the validation results to switch to R=16, tune alpha, choose the regional-head architecture, mine training negatives, calibrate a trained head, or otherwise optimize a model.

## Scientific question

Given the fixed EXP045 base-level ranking and the already-established R32 label oracle (**AP1 = 0.23831825797792802**), what combination of:

- **active-region recall**, and
- **suppression of dangerous truly-empty R32 regions**

is required to reach natural genome-wide nucleotide **AP1 = 0.20**?

## Frontier error model

This is a sensitivity analysis, not a trained model result and not a universal mathematical upper bound.

1. A region is strand-specific: `(contig, strand, position // 32)`.
2. Minus-track predictions are returned from reverse-complement order to genomic left→right **before** region assignment.
3. Empty R32 regions are ranked by **maximum EXP045 5-decimal nucleotide probability in that region**. Suppression proceeds from the highest-risk empty regions downward. This is deliberately optimistic and label-assisted: validation labels are used to know which regions are truly empty.
4. For active-region recall below 1.0, missed active regions are sampled uniformly at random from true active regions using several fixed seeds. The notebook reports median/min/max AP across these fixed random false-negative patterns.
5. Suppression moves all scores in a suppressed region to score bin 0, exactly matching the previous oracle convention. The local ordering of all retained positions is unchanged.

Therefore, for recall < 1.0 the resulting surface is a **pre-registered sensitivity frontier under this explicit error model**, not a proof that every gate with the same scalar recall/suppression pair must achieve the same AP.

## Integrity gates

Before the frontier is accepted, the notebook must reproduce:

- evaluated position-strand count = **83,807,486**;
- positive positions = **56,205**;
- EXP045 natural AP1 ≈ **0.09763009437871803**;
- R32 perfect-empty-suppression oracle AP1 ≈ **0.23831825797792802**;
- exact retention of all 56,205 positives in the perfect R32 oracle.

Any failure aborts the frontier.


In [1]:
from pathlib import Path
import gc
import hashlib
import json
import math
import os
import shutil
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch_directml

# ------------------------------------------------------------
# HARD ENVIRONMENT / PROJECT / NON-OVERWRITE GUARDS
# ------------------------------------------------------------
EXPECTED_PYTHON = os.path.normcase(os.path.abspath(
    r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe"
))
ACTUAL_PYTHON = os.path.normcase(os.path.abspath(sys.executable))
assert ACTUAL_PYTHON == EXPECTED_PYTHON, (
    f"Wrong interpreter: {sys.executable}\nExpected: {EXPECTED_PYTHON}"
)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src" / "ml_project").is_dir()
)
EXPECTED_PROJECT_ROOT = Path(r"D:\Projects\EPIC\EPIC").resolve()
assert os.path.normcase(str(PROJECT_ROOT)) == os.path.normcase(str(EXPECTED_PROJECT_ROOT)), (
    f"Wrong project root: {PROJECT_ROOT}"
)

AGENTS_PATH = PROJECT_ROOT.parent / "AGENTS.md"
assert AGENTS_PATH.is_file(), f"Missing required {AGENTS_PATH}"
AGENTS_SHA256 = hashlib.sha256(AGENTS_PATH.read_bytes()).hexdigest()
EXPECTED_AGENTS_SHA256_FROM_ORACLE = "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3"
assert AGENTS_SHA256 == EXPECTED_AGENTS_SHA256_FROM_ORACLE, (
    "AGENTS.md changed since the accepted oracle run. Review the new instructions before continuing.\n"
    f"Current:  {AGENTS_SHA256}\n"
    f"Oracle:   {EXPECTED_AGENTS_SHA256_FROM_ORACLE}"
)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

EXPERIMENT_ID = "CNN-EXP-046"
REGION_SIZE = 32
GRID_ORIGIN = 0
TARGET_AP = 0.20

RUN_DIR = (
    PROJECT_ROOT / "artifacts" / "experiments" /
    "CNN_EXP046_REGIONAL_ACTIVITY_GATE" / "run_001"
)
assert RUN_DIR.is_dir(), f"Oracle run directory does not exist: {RUN_DIR}"

FRONTIER_DIR = RUN_DIR / "feasibility_frontier_r32"
FRONTIER_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH_DIR = FRONTIER_DIR / "_scratch_region_cache"
assert not SCRATCH_DIR.exists(), (
    f"Scratch directory already exists: {SCRATCH_DIR}\n"
    "This may be an interrupted prior run. Inspect/remove it manually before restarting; "
    "the notebook will never overwrite it automatically."
)

ORACLE_JSON = RUN_DIR / "oracle_regional_gate_summary.json"
ORACLE_CSV = RUN_DIR / "oracle_regional_gate_results.csv"
FIXED_HARD_FP_CSV = RUN_DIR / "fixed_hard_fp_coordinates.csv"
FIXED_LOW_TP_CSV = RUN_DIR / "fixed_low_tp_coordinates.csv"
for p in (ORACLE_JSON, ORACLE_CSV, FIXED_HARD_FP_CSV, FIXED_LOW_TP_CSV):
    assert p.is_file(), f"Missing accepted oracle artifact: {p}"

OUTPUTS = {
    "prereg": FRONTIER_DIR / "EXP046_R32_frontier_preregistration.json",
    "raw_csv": FRONTIER_DIR / "r32_feasibility_frontier_raw.csv",
    "summary_csv": FRONTIER_DIR / "r32_feasibility_frontier_summary.csv",
    "crossing_csv": FRONTIER_DIR / "r32_target_ap_crossings.csv",
    "danger_csv": FRONTIER_DIR / "r32_empty_suppression_levels.csv",
    "summary_json": FRONTIER_DIR / "r32_feasibility_frontier_summary.json",
    "plot_lines": FRONTIER_DIR / "r32_frontier_ap_vs_hardfp_coverage.png",
    "plot_heatmap": FRONTIER_DIR / "r32_frontier_heatmap.png",
}
for name, path in OUTPUTS.items():
    if name != "prereg":
        assert not path.exists(), f"Refusing to overwrite existing output {name}: {path}"

# ------------------------------------------------------------
# FIXED FRONTIER GRID — diagnostic sensitivity grid, NOT tuning
# ------------------------------------------------------------
ACTIVE_REGION_RECALL_GRID = (
    1.000, 0.999, 0.995, 0.990, 0.980, 0.975, 0.950, 0.900
)
EMPTY_REGION_SUPPRESSION_GRID = (
    0.0000,
    0.0005, 0.0010, 0.0020, 0.0050,
    0.0100, 0.0200, 0.0300, 0.0500, 0.0750,
    0.1000, 0.1500, 0.2000, 0.3000, 0.4000,
    0.5000, 0.6000, 0.7000, 0.8000, 0.9000,
    0.9500, 0.9750, 0.9900, 1.0000,
)
FN_SEEDS = (4601, 4602, 4603, 4604, 4605)

EXPECTED_TOTAL_POSITION_STRAND = 83_807_486
EXPECTED_POSITIVES = 56_205
EXPECTED_BASELINE_AP = 0.09763009437871803
EXPECTED_ORACLE32_AP = 0.23831825797792802
EXPECTED_CONTIG_AP = {
    "NC_064034.1": 0.09569052125003817,
    "NC_064041.1": 0.09140733554044979,
    "NC_064042.1": 0.10520206334231234,
}
AP_TOL = 5e-6
SUBMISSION_LEVELS = 100_001
INVALID_SCORE_BIN = np.uint32(100_001)
CACHE_CHUNK_ROWS = 50_000
EVAL_BATCH_SIZE = 1
PROGRESS_EVERY = 250

EXP045_RUN_DIR = (
    PROJECT_ROOT / "artifacts" / "experiments" /
    "CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029" / "run_001"
)
CHECKPOINT_PATH = EXP045_RUN_DIR / "width512_50k_final.pt"
assert CHECKPOINT_PATH.is_file(), f"Missing EXP045 checkpoint: {CHECKPOINT_PATH}"

accepted_oracle = json.loads(ORACLE_JSON.read_text(encoding="utf-8"))
assert int(accepted_oracle["primary_region_size"]) == REGION_SIZE
assert abs(float(accepted_oracle["primary_oracle_ap1"]) - EXPECTED_ORACLE32_AP) <= 1e-12
assert abs(float(accepted_oracle["baseline_ap1"]) - EXPECTED_BASELINE_AP) <= 1e-12
assert bool(accepted_oracle["region_is_strand_specific"])
assert bool(accepted_oracle["minus_prediction_flipped_to_genomic_orientation_before_region_assignment"])

preregistration = {
    "experiment_id": EXPERIMENT_ID,
    "stage": "R32 feasibility frontier before frozen-head pilot",
    "region_size": REGION_SIZE,
    "region_key": "(contig, strand, floor(position / 32))",
    "grid_origin": GRID_ORIGIN,
    "checkpoint": str(CHECKPOINT_PATH),
    "target_nucleotide_ap1": TARGET_AP,
    "accepted_oracle32_ap1": EXPECTED_ORACLE32_AP,
    "active_region_recall_grid": list(ACTIVE_REGION_RECALL_GRID),
    "empty_region_suppression_grid": list(EMPTY_REGION_SUPPRESSION_GRID),
    "active_false_negative_seeds": list(FN_SEEDS),
    "danger_definition": "truly-empty R32 regions ordered by max EXP045 5dp nucleotide probability descending",
    "active_false_negative_model": "uniform random active-region drops; fixed seeds",
    "suppression_action": "move all nucleotide scores in suppressed region to 5dp score bin 0",
    "validation_use": "diagnostic feasibility only; no training/mining/tuning/calibration",
    "r16_selection_forbidden": True,
    "agents_sha256": AGENTS_SHA256,
}

if OUTPUTS["prereg"].exists():
    existing = json.loads(OUTPUTS["prereg"].read_text(encoding="utf-8"))
    assert existing == preregistration, (
        "Existing frontier preregistration differs from this notebook. "
        "Do not overwrite or silently change the frontier after seeing validation results."
    )
else:
    OUTPUTS["prereg"].write_text(
        json.dumps(preregistration, indent=2, ensure_ascii=False), encoding="utf-8"
    )

print("Project:", PROJECT_ROOT)
print("Interpreter:", sys.executable)
print("AGENTS.md SHA256:", AGENTS_SHA256)
print("Accepted Oracle32 AP1:", EXPECTED_ORACLE32_AP)
print("Frontier output dir:", FRONTIER_DIR)
print("Checkpoint:", CHECKPOINT_PATH)


Project: D:\Projects\EPIC\EPIC
Interpreter: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
AGENTS.md SHA256: bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3
Accepted Oracle32 AP1: 0.23831825797792802
Frontier output dir: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP046_REGIONAL_ACTIVITY_GATE\run_001\feasibility_frontier_r32
Checkpoint: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width512_50k_final.pt


In [2]:
# ------------------------------------------------------------
# LOAD DATA / MODEL / FIXED R32 VALIDATION GENERATOR
# ------------------------------------------------------------
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(split.intervals("train")["contig"].astype(str))
validation_contigs = set(split.intervals("validation")["contig"].astype(str))
assert train_contigs.isdisjoint(validation_contigs)
assert len(train_contigs) == 9
assert len(validation_contigs) == 3

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)
assert profile_config.target_length % REGION_SIZE == 0
REGIONS_PER_TILE = profile_config.target_length // REGION_SIZE
assert REGIONS_PER_TILE == 32

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
model = create_cnn_presence_intensity(channels=512)
model.load_state_dict(checkpoint["model_state_dict"])
assert int(checkpoint["step"]) == 50_000
assert int(model.backbone.stem.out_channels) == 512
assert int(model.profile_head.in_channels) == 512
assert int(model.profile_head.out_channels) == 2

device = torch_directml.device()
model = model.to(device)
model.eval()

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)
validation_tiles = validation_generator.tile_index.copy().reset_index(drop=True)
required_columns = {"contig", "target_start", "target_end", "allowed_position_strand"}
missing = required_columns - set(validation_tiles.columns)
assert not missing, f"tile_index missing columns: {sorted(missing)}"

assert int(validation_tiles["allowed_position_strand"].sum()) == EXPECTED_TOTAL_POSITION_STRAND
assert int(validation_generator.total_positive) == EXPECTED_POSITIVES
assert (validation_tiles["target_end"] - validation_tiles["target_start"] == profile_config.target_length).all()
assert (validation_tiles["target_start"].astype(np.int64) % REGION_SIZE == 0).all()
assert (validation_tiles["target_end"].astype(np.int64) % REGION_SIZE == 0).all()

for contig, frame in validation_tiles.groupby("contig", sort=True):
    frame = frame.sort_values("target_start")
    starts = frame["target_start"].to_numpy(dtype=np.int64)
    ends = frame["target_end"].to_numpy(dtype=np.int64)
    if len(frame) > 1:
        assert np.all(starts[1:] >= ends[:-1]), f"Overlapping target tiles on {contig}"

VALIDATION_CONTIGS = sorted(validation_contigs)
CONTIG_TO_CODE = {contig: i for i, contig in enumerate(VALIDATION_CONTIGS)}
assert len(CONTIG_TO_CODE) < 128

print("DirectML:", torch_directml.device_name(0))
print("Validation tiles:", len(validation_tiles))
print("Validation contigs:", VALIDATION_CONTIGS)
print("R32 regions/tile/strand:", REGIONS_PER_TILE)
print("Expected max region rows:", len(validation_tiles) * 2 * REGIONS_PER_TILE)

del checkpoint
gc.collect()


DirectML: AMD Radeon RX 6750 GRE 12GB 
Validation tiles: 47743
Validation contigs: ['NC_064034.1', 'NC_064041.1', 'NC_064042.1']
R32 regions/tile/strand: 32
Expected max region rows: 3055552


35

In [3]:
# ------------------------------------------------------------
# METRIC / REGION / CACHE HELPERS
# ------------------------------------------------------------
def sigmoid_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x, dtype=np.float64)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out.astype(np.float32)

def probability_bin_5dp(probabilities):
    return np.rint(np.clip(probabilities, 0.0, 1.0) * 100_000).astype(np.uint32)

def update_histogram(hist, indices):
    if len(indices) == 0:
        return
    values, counts = np.unique(indices, return_counts=True)
    hist[values.astype(np.int64)] += counts.astype(np.int64)

def ap_from_histograms(positive_hist, total_hist):
    positives = int(positive_hist.sum())
    assert positives > 0
    pos_desc = positive_hist[::-1]
    total_desc = total_hist[::-1]
    tp = np.cumsum(pos_desc, dtype=np.int64)
    predicted_positive = np.cumsum(total_desc, dtype=np.int64)
    precision = tp / np.maximum(predicted_positive, 1)
    return float(np.sum((pos_desc / positives) * precision))

BIT_SHIFTS = np.arange(REGION_SIZE, dtype=np.uint32)
BIT_WEIGHTS = np.left_shift(np.uint32(1), BIT_SHIFTS)
KEY_SHIFT = np.uint64(56)

def bool_rows_to_u32(x):
    x = np.asarray(x, dtype=bool)
    assert x.ndim == 2 and x.shape[1] == REGION_SIZE
    return np.sum(x.astype(np.uint32) * BIT_WEIGHTS[None, :], axis=1, dtype=np.uint32)

def u32_to_bool_rows(mask):
    mask = np.asarray(mask, dtype=np.uint32)
    return ((mask[:, None] >> BIT_SHIFTS[None, :]) & np.uint32(1)).astype(bool)

def pack_region_keys(contig_code, strand_code, region_ids):
    prefix = np.uint64(int(contig_code) * 2 + int(strand_code)) << KEY_SHIFT
    region_ids = np.asarray(region_ids, dtype=np.uint64)
    assert np.all(region_ids < (np.uint64(1) << KEY_SHIFT))
    return prefix | region_ids

def dataframe_region_keys(frame):
    contig_code = frame["contig"].astype(str).map(CONTIG_TO_CODE)
    assert contig_code.notna().all(), "Cohort contains a contig outside validation"
    strand = frame["strand"].astype(str)
    assert strand.isin(["+", "-"]).all()
    strand_code = (strand == "-").to_numpy(dtype=np.uint64)
    prefix_code = contig_code.to_numpy(dtype=np.uint64) * np.uint64(2) + strand_code
    positions = frame["position"].to_numpy(dtype=np.int64)
    assert np.all(positions >= 0)
    region_id = ((positions - GRID_ORIGIN) // REGION_SIZE).astype(np.uint64)
    return (prefix_code << KEY_SHIFT) | region_id

def move_suppressed_hist_to_zero(baseline_total, baseline_pos, removed_total, removed_pos):
    total = baseline_total.astype(np.int64, copy=True) - removed_total.astype(np.int64, copy=False)
    pos = baseline_pos.astype(np.int64, copy=True) - removed_pos.astype(np.int64, copy=False)
    assert np.all(total >= 0)
    assert np.all(pos >= 0)
    total[0] += int(removed_total.sum())
    pos[0] += int(removed_pos.sum())
    assert int(total.sum()) == EXPECTED_TOTAL_POSITION_STRAND
    assert int(pos.sum()) == EXPECTED_POSITIVES
    return total, pos

print("Helper definitions loaded.")


Helper definitions loaded.


## 1. One inference pass: build an R32 scalar-score cache

The previous oracle artifacts intentionally did not save all genome-wide predictions. To evaluate many frontier scenarios without repeatedly running the CNN, this notebook performs **one** full EXP045 validation inference pass and writes a temporary cache containing only:

- the 5-decimal scalar EXP045 score bin for each base;
- a 32-bit valid-position mask per R32 region;
- a 32-bit positive-position mask per R32 region;
- one packed integer region key;
- one max-score danger value per region.

It does **not** save a 512-D genome-wide feature tensor. The temporary cache is deleted after the frontier outputs are written.


In [4]:
# ------------------------------------------------------------
# BUILD TEMPORARY R32 REGION CACHE + REPRODUCE EXP045
# ------------------------------------------------------------
SCRATCH_DIR.mkdir(parents=False, exist_ok=False)
MAX_REGION_ROWS = len(validation_tiles) * 2 * REGIONS_PER_TILE

score_path = SCRATCH_DIR / "score_bins_u32.dat"
valid_mask_path = SCRATCH_DIR / "valid_mask_u32.dat"
positive_mask_path = SCRATCH_DIR / "positive_mask_u32.dat"
region_key_path = SCRATCH_DIR / "region_key_u64.dat"
risk_path = SCRATCH_DIR / "region_max_score_u32.dat"

score_mm = np.memmap(score_path, mode="w+", dtype=np.uint32, shape=(MAX_REGION_ROWS, REGION_SIZE))
valid_mask_mm = np.memmap(valid_mask_path, mode="w+", dtype=np.uint32, shape=(MAX_REGION_ROWS,))
positive_mask_mm = np.memmap(positive_mask_path, mode="w+", dtype=np.uint32, shape=(MAX_REGION_ROWS,))
region_key_mm = np.memmap(region_key_path, mode="w+", dtype=np.uint64, shape=(MAX_REGION_ROWS,))
risk_mm = np.memmap(risk_path, mode="w+", dtype=np.uint32, shape=(MAX_REGION_ROWS,))

segment_names = ["overall", *VALIDATION_CONTIGS]
baseline_total_hist = {name: np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for name in segment_names}
baseline_pos_hist = {name: np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for name in segment_names}

row_cursor = 0
positions_seen = 0
positives_seen = 0
started = time.perf_counter()

with torch.no_grad():
    for batch_no, start in enumerate(range(0, len(validation_tiles), EVAL_BATCH_SIZE), 1):
        selected = validation_tiles.iloc[start:start + EVAL_BATCH_SIZE]
        batch = validation_generator.make_batch(selected)
        B = len(selected)
        x_gpu = batch.x_both_strands.to(device)

        plus_presence_gpu, minus_presence_reverse_gpu, _, _ = forward_both_strands_multitask(
            model,
            x_gpu,
            batch_size=B,
            context_flank=profile_config.context_flank,
            target_length=profile_config.target_length,
        )

        plus_probability = sigmoid_numpy(
            plus_presence_gpu[:, 0, :].detach().cpu().numpy().astype(np.float32, copy=True)
        )
        minus_probability = sigmoid_numpy(
            minus_presence_reverse_gpu[:, 0, :].detach().cpu().numpy().astype(np.float32, copy=True)
        )[:, ::-1].copy()  # RC -> genomic left-to-right BEFORE region IDs

        del x_gpu, plus_presence_gpu, minus_presence_reverse_gpu

        plus_count = batch.count[:, 0, :].numpy().astype(np.float32, copy=False)
        minus_count = batch.count[:, 1, :].numpy().astype(np.float32, copy=False)
        plus_mask = batch.mask[:, 0, :].numpy().astype(bool, copy=False)
        minus_mask = batch.mask[:, 1, :].numpy().astype(bool, copy=False)

        for local_i in range(B):
            row = selected.iloc[local_i]
            contig = str(row["contig"])
            contig_code = CONTIG_TO_CODE[contig]
            target_start = int(row["target_start"])
            target_end = int(row["target_end"])
            assert target_end - target_start == profile_config.target_length
            assert target_start % REGION_SIZE == 0

            first_region_id = (target_start - GRID_ORIGIN) // REGION_SIZE
            region_ids = first_region_id + np.arange(REGIONS_PER_TILE, dtype=np.int64)

            for strand_code, probability, count, mask in (
                (0, plus_probability[local_i], plus_count[local_i], plus_mask[local_i]),
                (1, minus_probability[local_i], minus_count[local_i], minus_mask[local_i]),
            ):
                score_bins = probability_bin_5dp(probability)
                positive = (count > 0) & mask

                score_2d = score_bins.reshape(REGIONS_PER_TILE, REGION_SIZE)
                valid_2d = mask.reshape(REGIONS_PER_TILE, REGION_SIZE)
                positive_2d = positive.reshape(REGIONS_PER_TILE, REGION_SIZE)
                assert np.all(~positive_2d | valid_2d)

                sl = slice(row_cursor, row_cursor + REGIONS_PER_TILE)
                score_mm[sl, :] = score_2d
                valid_mask_mm[sl] = bool_rows_to_u32(valid_2d)
                positive_mask_mm[sl] = bool_rows_to_u32(positive_2d)
                region_key_mm[sl] = pack_region_keys(contig_code, strand_code, region_ids)
                risk_mm[sl] = np.where(valid_2d, score_2d, np.uint32(0)).max(axis=1).astype(np.uint32)
                row_cursor += REGIONS_PER_TILE

                valid_bins = score_bins[mask]
                pos_bins = score_bins[positive]
                positions_seen += int(mask.sum())
                positives_seen += int(positive.sum())
                for segment in ("overall", contig):
                    update_histogram(baseline_total_hist[segment], valid_bins)
                    update_histogram(baseline_pos_hist[segment], pos_bins)

        del (
            batch, selected, plus_probability, minus_probability,
            plus_count, minus_count, plus_mask, minus_mask,
        )
        if batch_no % 25 == 0:
            gc.collect()
        if batch_no == 1 or batch_no % PROGRESS_EVERY == 0 or start + B >= len(validation_tiles):
            elapsed = time.perf_counter() - started
            print(
                f"tiles={start + B:>6}/{len(validation_tiles)} "
                f"positions={positions_seen:>12,} positives={positives_seen:>7,} "
                f"regions={row_cursor:>9,} positions/s={positions_seen / elapsed:,.0f}"
            )

for mm in (score_mm, valid_mask_mm, positive_mask_mm, region_key_mm, risk_mm):
    mm.flush()

assert row_cursor == MAX_REGION_ROWS
assert positions_seen == EXPECTED_TOTAL_POSITION_STRAND
assert positives_seen == EXPECTED_POSITIVES

baseline_ap = {
    segment: ap_from_histograms(baseline_pos_hist[segment], baseline_total_hist[segment])
    for segment in segment_names
}
assert abs(baseline_ap["overall"] - EXPECTED_BASELINE_AP) <= AP_TOL, baseline_ap["overall"]
for contig, expected in EXPECTED_CONTIG_AP.items():
    assert abs(baseline_ap[contig] - expected) <= AP_TOL, (contig, baseline_ap[contig], expected)

print("EXP045 reproduction: PASS")
print(json.dumps(baseline_ap, indent=2))
print("Temporary scalar cache size (GiB):", sum(p.stat().st_size for p in SCRATCH_DIR.iterdir()) / (1024 ** 3))


tiles=     1/47743 positions=         210 positives=      0 regions=       64 positions/s=1,571
tiles=   250/47743 positions=     326,316 positives=     11 regions=   16,000 positions/s=52,464
tiles=   500/47743 positions=     708,204 positives=    445 regions=   32,000 positions/s=57,313
tiles=   750/47743 positions=   1,102,632 positives=    790 regions=   48,000 positions/s=59,770
tiles=  1000/47743 positions=   1,519,462 positives=  1,034 regions=   64,000 positions/s=61,946
tiles=  1250/47743 positions=   1,928,284 positives=  1,209 regions=   80,000 positions/s=63,003
tiles=  1500/47743 positions=   2,381,142 positives=  1,626 regions=   96,000 positions/s=64,857
tiles=  1750/47743 positions=   2,825,034 positives=  1,980 regions=  112,000 positions/s=66,005
tiles=  2000/47743 positions=   3,270,676 positives=  2,259 regions=  128,000 positions/s=66,915
tiles=  2250/47743 positions=   3,688,504 positives=  2,496 regions=  144,000 positions/s=67,065
tiles=  2500/47743 positions=  

In [5]:
# ------------------------------------------------------------
# REGION AUDIT + FIXED COHORT MAPPING
# ------------------------------------------------------------
all_rows = np.arange(row_cursor, dtype=np.int64)
valid_region_rows = all_rows[np.asarray(valid_mask_mm[:row_cursor]) != 0]
active_rows = valid_region_rows[np.asarray(positive_mask_mm[valid_region_rows]) != 0]
empty_rows = valid_region_rows[np.asarray(positive_mask_mm[valid_region_rows]) == 0]

assert len(active_rows) + len(empty_rows) == len(valid_region_rows)
assert len(active_rows) > 0 and len(empty_rows) > 0

# Region keys must be unique. If not, target tiles overlap the same genomic R32 region.
keys_valid = np.asarray(region_key_mm[valid_region_rows], dtype=np.uint64)
key_order = np.argsort(keys_valid, kind="mergesort")
sorted_keys = keys_valid[key_order]
assert np.all(sorted_keys[1:] > sorted_keys[:-1]), "Duplicate R32 region keys found"
sorted_rows = valid_region_rows[key_order]

def lookup_region_rows(keys):
    keys = np.asarray(keys, dtype=np.uint64)
    loc = np.searchsorted(sorted_keys, keys)
    assert np.all(loc < len(sorted_keys)), "Cohort region key outside cached validation regions"
    assert np.array_equal(sorted_keys[loc], keys), "Cohort region key not found exactly"
    return sorted_rows[loc]

hard_fp_df = pd.read_csv(FIXED_HARD_FP_CSV)
low_tp_df = pd.read_csv(FIXED_LOW_TP_CSV)
assert len(hard_fp_df) == 20_000
assert len(low_tp_df) == 14_052

hard_fp_rows = lookup_region_rows(dataframe_region_keys(hard_fp_df))
low_tp_rows = lookup_region_rows(dataframe_region_keys(low_tp_df))

hard_fp_region_is_empty = np.asarray(positive_mask_mm[hard_fp_rows]) == 0
hard_fp_empty_coordinate_rows = hard_fp_rows[hard_fp_region_is_empty]
hard_fp_empty_unique_rows = np.unique(hard_fp_empty_coordinate_rows)

# Reproduce accepted R32 HARD_FP diagnostic exactly at coordinate level.
assert int(hard_fp_region_is_empty.sum()) == round(20_000 * 0.7921), (
    int(hard_fp_region_is_empty.sum()), 20_000 * 0.7921
)
assert np.all(np.asarray(positive_mask_mm[low_tp_rows]) != 0), "LOW_TP must belong to active R32 regions"

region_stats = {
    "valid_regions": int(len(valid_region_rows)),
    "active_regions": int(len(active_rows)),
    "empty_regions": int(len(empty_rows)),
    "active_region_prevalence": float(len(active_rows) / len(valid_region_rows)),
    "fixed_hard_fp_coordinates": int(len(hard_fp_df)),
    "fixed_hard_fp_coordinates_in_empty_r32": int(len(hard_fp_empty_coordinate_rows)),
    "fixed_hard_fp_empty_fraction": float(len(hard_fp_empty_coordinate_rows) / len(hard_fp_df)),
    "unique_empty_r32_regions_containing_fixed_hard_fp": int(len(hard_fp_empty_unique_rows)),
    "fixed_low_tp_coordinates": int(len(low_tp_df)),
}
print(json.dumps(region_stats, indent=2))

del keys_valid, key_order, sorted_keys, sorted_rows
gc.collect()


{
  "valid_regions": 2675614,
  "active_regions": 28303,
  "empty_regions": 2647311,
  "active_region_prevalence": 0.010578132720190581,
  "fixed_hard_fp_coordinates": 20000,
  "fixed_hard_fp_coordinates_in_empty_r32": 15842,
  "fixed_hard_fp_empty_fraction": 0.7921,
  "unique_empty_r32_regions_containing_fixed_hard_fp": 12986,
  "fixed_low_tp_coordinates": 14052
}


72

## 2. Danger-ranked empty-region suppression

For the frontier, “dangerous empty region” is fixed **before** seeing the frontier:

> a truly-empty R32 region with a high **maximum EXP045 nucleotide score**.

Empty regions are sorted by that score descending. Ties are broken deterministically by packed genomic region key.

For every pre-registered suppression fraction, the notebook stores the cumulative histogram of base scores removed from those empty regions. It also reports how much of the **fixed EXP045 HARD_FP cohort** lies in the suppressed region set.


In [6]:
# ------------------------------------------------------------
# AGGREGATE SCORE HISTOGRAMS FOR ARBITRARY REGION SETS
# ------------------------------------------------------------
def aggregate_region_rows(rows, need_positive=True):
    rows = np.asarray(rows, dtype=np.int64)
    total_hist = np.zeros(SUBMISSION_LEVELS, dtype=np.int64)
    pos_hist = np.zeros(SUBMISSION_LEVELS, dtype=np.int64)
    total_n = 0
    pos_n = 0

    for start in range(0, len(rows), CACHE_CHUNK_ROWS):
        chunk = rows[start:start + CACHE_CHUNK_ROWS]
        bins = np.asarray(score_mm[chunk, :], dtype=np.uint32)
        valid = u32_to_bool_rows(np.asarray(valid_mask_mm[chunk], dtype=np.uint32))
        valid_bins = bins[valid]
        total_hist += np.bincount(valid_bins.astype(np.int64), minlength=SUBMISSION_LEVELS).astype(np.int64)
        total_n += int(valid.sum())

        if need_positive:
            pos = u32_to_bool_rows(np.asarray(positive_mask_mm[chunk], dtype=np.uint32))
            assert np.all(~pos | valid)
            pos_bins = bins[pos]
            pos_hist += np.bincount(pos_bins.astype(np.int64), minlength=SUBMISSION_LEVELS).astype(np.int64)
            pos_n += int(pos.sum())

    assert int(total_hist.sum()) == total_n
    assert int(pos_hist.sum()) == pos_n
    return total_hist, pos_hist, total_n, pos_n

# Danger order: max EXP045 score DESC, deterministic genomic key ASC on ties.
empty_risk = np.asarray(risk_mm[empty_rows], dtype=np.int64)
empty_keys = np.asarray(region_key_mm[empty_rows], dtype=np.uint64)
empty_order_local = np.lexsort((empty_keys, -empty_risk))
empty_sorted = empty_rows[empty_order_local]
assert len(empty_sorted) == len(empty_rows)

# Rank lookup is needed only for fixed HARD_FP coverage diagnostics.
empty_rank = np.full(row_cursor, -1, dtype=np.int32)
empty_rank[empty_sorted] = np.arange(len(empty_sorted), dtype=np.int32)
assert np.all(empty_rank[hard_fp_empty_coordinate_rows] >= 0)

empty_removed_hist_by_fraction = {}
empty_removed_n_by_fraction = {}
danger_rows = []

cumulative_hist = np.zeros(SUBMISSION_LEVELS, dtype=np.int64)
cumulative_n = 0
previous_target = 0

for fraction in EMPTY_REGION_SUPPRESSION_GRID:
    target = int(round(float(fraction) * len(empty_sorted)))
    target = min(max(target, 0), len(empty_sorted))
    if fraction == 1.0:
        target = len(empty_sorted)
    assert target >= previous_target

    if target > previous_target:
        inc_rows = empty_sorted[previous_target:target]
        inc_total, inc_pos, inc_n, inc_pos_n = aggregate_region_rows(inc_rows, need_positive=False)
        assert inc_pos_n == 0 and int(inc_pos.sum()) == 0
        cumulative_hist += inc_total
        cumulative_n += inc_n

    empty_removed_hist_by_fraction[float(fraction)] = cumulative_hist.copy()
    empty_removed_n_by_fraction[float(fraction)] = int(cumulative_n)

    coord_coverage = (
        float(np.mean(empty_rank[hard_fp_empty_coordinate_rows] < target))
        if len(hard_fp_empty_coordinate_rows) else float("nan")
    )
    region_coverage = (
        float(np.mean(empty_rank[hard_fp_empty_unique_rows] < target))
        if len(hard_fp_empty_unique_rows) else float("nan")
    )
    danger_threshold_bin = int(risk_mm[empty_sorted[target - 1]]) if target > 0 else None

    danger_rows.append({
        "empty_region_suppression_fraction_target": float(fraction),
        "empty_region_suppression_fraction_realized": float(target / len(empty_sorted)),
        "suppressed_empty_regions": int(target),
        "suppressed_empty_positions": int(cumulative_n),
        "danger_threshold_max_exp045_score": (
            danger_threshold_bin / 100_000.0 if danger_threshold_bin is not None else np.nan
        ),
        "fixed_hard_fp_empty_coordinate_coverage": coord_coverage,
        "fixed_hard_fp_empty_region_coverage": region_coverage,
    })
    previous_target = target
    print(
        f"empty suppression={fraction:>7.4f} regions={target:>9,} "
        f"HARD_FP-empty coordinate coverage={coord_coverage:>7.3%}"
    )

danger_df = pd.DataFrame(danger_rows)

# Perfect empty suppression must exactly reproduce the accepted R32 oracle.
full_empty_hist = empty_removed_hist_by_fraction[1.0]
zero_pos_removed = np.zeros(SUBMISSION_LEVELS, dtype=np.int64)
oracle_total, oracle_pos = move_suppressed_hist_to_zero(
    baseline_total_hist["overall"],
    baseline_pos_hist["overall"],
    full_empty_hist,
    zero_pos_removed,
)
oracle32_ap = ap_from_histograms(oracle_pos, oracle_total)
assert int(oracle_pos.sum()) == EXPECTED_POSITIVES
assert abs(oracle32_ap - EXPECTED_ORACLE32_AP) <= AP_TOL, (oracle32_ap, EXPECTED_ORACLE32_AP)

print("R32 oracle reproduction: PASS")
print("Oracle32 AP1:", oracle32_ap)
display(danger_df)

del empty_risk, empty_keys, empty_order_local
gc.collect()


empty suppression= 0.0000 regions=        0 HARD_FP-empty coordinate coverage= 0.000%
empty suppression= 0.0005 regions=    1,324 HARD_FP-empty coordinate coverage=10.352%
empty suppression= 0.0010 regions=    2,647 HARD_FP-empty coordinate coverage=18.811%
empty suppression= 0.0020 regions=    5,295 HARD_FP-empty coordinate coverage=34.320%
empty suppression= 0.0050 regions=   13,237 HARD_FP-empty coordinate coverage=73.949%
empty suppression= 0.0100 regions=   26,473 HARD_FP-empty coordinate coverage=100.000%
empty suppression= 0.0200 regions=   52,946 HARD_FP-empty coordinate coverage=100.000%
empty suppression= 0.0300 regions=   79,419 HARD_FP-empty coordinate coverage=100.000%
empty suppression= 0.0500 regions=  132,366 HARD_FP-empty coordinate coverage=100.000%
empty suppression= 0.0750 regions=  198,548 HARD_FP-empty coordinate coverage=100.000%
empty suppression= 0.1000 regions=  264,731 HARD_FP-empty coordinate coverage=100.000%
empty suppression= 0.1500 regions=  397,097 HARD

,empty_region_suppression_fraction_target,empty_region_suppression_fraction_realized,suppressed_empty_regions,suppressed_empty_positions,danger_threshold_max_exp045_score,fixed_hard_fp_empty_coordinate_coverage,fixed_hard_fp_empty_region_coverage
0,0.0000,0.0000,0,0,NaN,0.000000,0.000000
1,0.0005,0.0005,1324,42082,0.99973,0.103522,0.084014
2,0.0010,0.0010,2647,84080,0.99947,0.188108,0.158093
3,0.0020,0.0020,5295,168130,0.99896,0.343202,0.301787
4,0.0050,0.0050,13237,420072,0.99740,0.739490,0.701525
5,0.0100,0.0100,26473,840176,0.99472,1.000000,1.000000
6,0.0200,0.0200,52946,1680485,0.98929,1.000000,1.000000
7,0.0300,0.0300,79419,2520559,0.98347,1.000000,1.000000
8,0.0500,0.0500,132366,4201473,0.97090,1.000000,1.000000
9,0.0750,0.0750,198548,6301034,0.95268,1.000000,1.000000


0

## 3. Active-region recall sensitivity

For recall below 1.0, the notebook does **not** choose which active regions to miss using validation performance. Instead it uses five fixed random permutations of true active R32 regions.

This gives a distribution of AP for the same scalar active-region recall. We additionally report:

- realized **positive-position recall**;
- fixed **LOW_TP retention**;
- fixed HARD_FP-empty coordinate/region coverage from the danger-ranked empty suppression.

The important distinction is that active-region recall and positive-position recall are not identical when a region contains more than one positive nucleotide.


In [7]:
# ------------------------------------------------------------
# BUILD THE FEASIBILITY FRONTIER
# ------------------------------------------------------------
danger_lookup = danger_df.set_index("empty_region_suppression_fraction_target").to_dict(orient="index")
frontier_rows = []

for seed in FN_SEEDS:
    rng = np.random.default_rng(seed)
    active_permutation = rng.permutation(active_rows)

    # Map every active region to its fixed random-drop rank for LOW_TP retention.
    active_drop_rank = np.full(row_cursor, -1, dtype=np.int32)
    active_drop_rank[active_permutation] = np.arange(len(active_permutation), dtype=np.int32)
    assert np.all(active_drop_rank[low_tp_rows] >= 0)

    cumulative_active_total = np.zeros(SUBMISSION_LEVELS, dtype=np.int64)
    cumulative_active_pos = np.zeros(SUBMISSION_LEVELS, dtype=np.int64)
    cumulative_active_n = 0
    cumulative_active_pos_n = 0
    previous_drop_n = 0

    # Recall decreases -> number of dropped active regions increases monotonically.
    for recall_target in ACTIVE_REGION_RECALL_GRID:
        drop_n = int(round((1.0 - float(recall_target)) * len(active_rows)))
        drop_n = min(max(drop_n, 0), len(active_rows))
        assert drop_n >= previous_drop_n

        if drop_n > previous_drop_n:
            inc_rows = active_permutation[previous_drop_n:drop_n]
            inc_total, inc_pos, inc_n, inc_pos_n = aggregate_region_rows(inc_rows, need_positive=True)
            cumulative_active_total += inc_total
            cumulative_active_pos += inc_pos
            cumulative_active_n += inc_n
            cumulative_active_pos_n += inc_pos_n

        active_recall_realized = 1.0 - drop_n / len(active_rows)
        positive_position_recall = 1.0 - cumulative_active_pos_n / EXPECTED_POSITIVES
        low_tp_retention = float(np.mean(active_drop_rank[low_tp_rows] >= drop_n))

        for empty_fraction in EMPTY_REGION_SUPPRESSION_GRID:
            empty_fraction = float(empty_fraction)
            removed_total = (
                empty_removed_hist_by_fraction[empty_fraction] + cumulative_active_total
            )
            removed_pos = cumulative_active_pos
            scenario_total, scenario_pos = move_suppressed_hist_to_zero(
                baseline_total_hist["overall"],
                baseline_pos_hist["overall"],
                removed_total,
                removed_pos,
            )
            ap1 = ap_from_histograms(scenario_pos, scenario_total)

            d = danger_lookup[empty_fraction]
            frontier_rows.append({
                "seed": int(seed),
                "active_region_recall_target": float(recall_target),
                "active_region_recall_realized": float(active_recall_realized),
                "dropped_active_regions": int(drop_n),
                "positive_position_recall": float(positive_position_recall),
                "low_tp_retention": float(low_tp_retention),
                "empty_region_suppression_fraction_target": empty_fraction,
                "empty_region_suppression_fraction_realized": float(d["empty_region_suppression_fraction_realized"]),
                "suppressed_empty_regions": int(d["suppressed_empty_regions"]),
                "danger_threshold_max_exp045_score": float(d["danger_threshold_max_exp045_score"]),
                "fixed_hard_fp_empty_coordinate_coverage": float(d["fixed_hard_fp_empty_coordinate_coverage"]),
                "fixed_hard_fp_empty_region_coverage": float(d["fixed_hard_fp_empty_region_coverage"]),
                "nucleotide_ap1": float(ap1),
            })

        previous_drop_n = drop_n

    del active_drop_rank, active_permutation
    gc.collect()
    print(f"seed {seed}: complete")

frontier_raw = pd.DataFrame(frontier_rows)

# Exact deterministic corner checks.
corner = frontier_raw[
    (frontier_raw["active_region_recall_target"] == 1.0) &
    (frontier_raw["empty_region_suppression_fraction_target"] == 1.0)
]
assert len(corner) == len(FN_SEEDS)
assert np.allclose(corner["nucleotide_ap1"].to_numpy(), EXPECTED_ORACLE32_AP, atol=AP_TOL, rtol=0)
assert np.all(corner["positive_position_recall"].to_numpy() == 1.0)
assert np.all(corner["low_tp_retention"].to_numpy() == 1.0)

baseline_corner = frontier_raw[
    (frontier_raw["active_region_recall_target"] == 1.0) &
    (frontier_raw["empty_region_suppression_fraction_target"] == 0.0)
]
assert np.allclose(baseline_corner["nucleotide_ap1"].to_numpy(), EXPECTED_BASELINE_AP, atol=AP_TOL, rtol=0)

print("Frontier corner invariants: PASS")
print("Raw scenarios:", len(frontier_raw))


seed 4601: complete
seed 4602: complete
seed 4603: complete
seed 4604: complete
seed 4605: complete
Frontier corner invariants: PASS
Raw scenarios: 960


In [8]:
# ------------------------------------------------------------
# SUMMARIZE FRONTIER + FIRST GRID CROSSINGS OF AP1=0.20
# ------------------------------------------------------------
frontier_summary = (
    frontier_raw
    .groupby(
        ["active_region_recall_target", "empty_region_suppression_fraction_target"],
        as_index=False,
        sort=True,
    )
    .agg(
        active_region_recall_realized=("active_region_recall_realized", "first"),
        median_ap1=("nucleotide_ap1", "median"),
        min_ap1=("nucleotide_ap1", "min"),
        max_ap1=("nucleotide_ap1", "max"),
        mean_ap1=("nucleotide_ap1", "mean"),
        median_positive_position_recall=("positive_position_recall", "median"),
        min_positive_position_recall=("positive_position_recall", "min"),
        median_low_tp_retention=("low_tp_retention", "median"),
        min_low_tp_retention=("low_tp_retention", "min"),
        empty_region_suppression_fraction_realized=("empty_region_suppression_fraction_realized", "first"),
        fixed_hard_fp_empty_coordinate_coverage=("fixed_hard_fp_empty_coordinate_coverage", "first"),
        fixed_hard_fp_empty_region_coverage=("fixed_hard_fp_empty_region_coverage", "first"),
        danger_threshold_max_exp045_score=("danger_threshold_max_exp045_score", "first"),
    )
)

def first_crossing(frame, value_column):
    hit = frame[frame[value_column] >= TARGET_AP].sort_values(
        "empty_region_suppression_fraction_target"
    )
    if hit.empty:
        return None
    return hit.iloc[0]

crossing_rows = []
for recall_target in ACTIVE_REGION_RECALL_GRID:
    sub = frontier_summary[
        frontier_summary["active_region_recall_target"] == float(recall_target)
    ].copy()
    median_hit = first_crossing(sub, "median_ap1")
    robust_hit = first_crossing(sub, "min_ap1")

    row = {
        "active_region_recall_target": float(recall_target),
        "median_crossing_found": median_hit is not None,
        "robust_all_seeds_crossing_found": robust_hit is not None,
    }
    for prefix, hit in (("median", median_hit), ("robust", robust_hit)):
        if hit is None:
            row[f"{prefix}_first_empty_suppression_fraction"] = np.nan
            row[f"{prefix}_hard_fp_empty_coordinate_coverage"] = np.nan
            row[f"{prefix}_hard_fp_empty_region_coverage"] = np.nan
            row[f"{prefix}_positive_position_recall"] = np.nan
            row[f"{prefix}_low_tp_retention"] = np.nan
            row[f"{prefix}_ap1"] = np.nan
        else:
            row[f"{prefix}_first_empty_suppression_fraction"] = float(hit["empty_region_suppression_fraction_realized"])
            row[f"{prefix}_hard_fp_empty_coordinate_coverage"] = float(hit["fixed_hard_fp_empty_coordinate_coverage"])
            row[f"{prefix}_hard_fp_empty_region_coverage"] = float(hit["fixed_hard_fp_empty_region_coverage"])
            row[f"{prefix}_positive_position_recall"] = float(
                hit["median_positive_position_recall"] if prefix == "median" else hit["min_positive_position_recall"]
            )
            row[f"{prefix}_low_tp_retention"] = float(
                hit["median_low_tp_retention"] if prefix == "median" else hit["min_low_tp_retention"]
            )
            row[f"{prefix}_ap1"] = float(hit["median_ap1"] if prefix == "median" else hit["min_ap1"])
    crossing_rows.append(row)

crossings = pd.DataFrame(crossing_rows)

print("First pre-registered grid points reaching AP1 >= 0.20")
display(crossings)

# Focused table near the operationally important high-recall range.
focus = frontier_summary[
    frontier_summary["active_region_recall_target"] >= 0.95
].copy()
focus["target_reached_median"] = focus["median_ap1"] >= TARGET_AP
focus["target_reached_all_seeds"] = focus["min_ap1"] >= TARGET_AP
display(focus)


First pre-registered grid points reaching AP1 >= 0.20


,active_region_recall_target,median_crossing_found,robust_all_seeds_crossing_found,median_first_empty_suppression_fraction,median_hard_fp_empty_coordinate_coverage,median_hard_fp_empty_region_coverage,median_positive_position_recall,median_low_tp_retention,median_ap1,robust_first_empty_suppression_fraction,robust_hard_fp_empty_coordinate_coverage,robust_hard_fp_empty_region_coverage,robust_positive_position_recall,robust_low_tp_retention,robust_ap1
0,1.000,True,True,0.10,1.0,1.0,1.000000,1.000000,0.204801,0.10,1.0,1.0,1.000000,1.000000,0.204801
1,0.999,True,True,0.10,1.0,1.0,0.999146,0.999004,0.204611,0.10,1.0,1.0,0.998755,0.998577,0.204449
2,0.995,True,True,0.10,1.0,1.0,0.995534,0.994876,0.203843,0.10,1.0,1.0,0.994040,0.994449,0.203016
3,0.990,True,True,0.10,1.0,1.0,0.991033,0.990393,0.203089,0.10,1.0,1.0,0.988595,0.988614,0.201901
4,0.980,True,True,0.10,1.0,1.0,0.980838,0.979149,0.200937,0.15,1.0,1.0,0.977155,0.977512,0.207841
5,0.975,True,True,0.15,1.0,1.0,0.975500,0.975021,0.208114,0.15,1.0,1.0,0.972244,0.972602,0.207027
6,0.950,True,True,0.15,1.0,1.0,0.951428,0.951324,0.202123,0.15,1.0,1.0,0.948012,0.947694,0.201598
7,0.900,True,True,0.30,1.0,1.0,0.901094,0.899089,0.203599,0.30,1.0,1.0,0.898141,0.896883,0.202967


,active_region_recall_target,empty_region_suppression_fraction_target,active_region_recall_realized,median_ap1,min_ap1,max_ap1,mean_ap1,median_positive_position_recall,min_positive_position_recall,median_low_tp_retention,min_low_tp_retention,empty_region_suppression_fraction_realized,fixed_hard_fp_empty_coordinate_coverage,fixed_hard_fp_empty_region_coverage,danger_threshold_max_exp045_score,target_reached_median,target_reached_all_seeds
24,0.95,0.0000,0.950005,0.090755,0.090077,0.091087,0.090617,0.951428,0.948012,0.951324,0.947694,0.0000,0.000000,0.000000,NaN,False,False
25,0.95,0.0005,0.950005,0.111162,0.110300,0.111421,0.110937,0.951428,0.948012,0.951324,0.947694,0.0005,0.103522,0.084014,0.99973,False,False
26,0.95,0.0010,0.950005,0.117927,0.116983,0.118115,0.117626,0.951428,0.948012,0.951324,0.947694,0.0010,0.188108,0.158093,0.99947,False,False
27,0.95,0.0020,0.950005,0.125699,0.124727,0.125850,0.125363,0.951428,0.948012,0.951324,0.947694,0.0020,0.343202,0.301787,0.99896,False,False
28,0.95,0.0050,0.950005,0.137717,0.136822,0.137949,0.137446,0.951428,0.948012,0.951324,0.947694,0.0050,0.739490,0.701525,0.99740,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,1.00,0.9000,1.000000,0.237964,0.237964,0.237964,0.237964,1.000000,1.000000,1.000000,1.000000,0.9000,1.000000,1.000000,0.02828,True,True
188,1.00,0.9500,1.000000,0.238198,0.238198,0.238198,0.238198,1.000000,1.000000,1.000000,1.000000,0.9500,1.000000,1.000000,0.01277,True,True
189,1.00,0.9750,1.000000,0.238282,0.238282,0.238282,0.238282,1.000000,1.000000,1.000000,1.000000,0.9750,1.000000,1.000000,0.00629,True,True
190,1.00,0.9900,1.000000,0.238312,0.238312,0.238312,0.238312,1.000000,1.000000,1.000000,1.000000,0.9900,1.000000,1.000000,0.00254,True,True


In [9]:
# ------------------------------------------------------------
# SAVE TABLES / PLOTS / MACHINE-READABLE SUMMARY
# ------------------------------------------------------------
frontier_raw.to_csv(OUTPUTS["raw_csv"], index=False)
frontier_summary.to_csv(OUTPUTS["summary_csv"], index=False)
crossings.to_csv(OUTPUTS["crossing_csv"], index=False)
danger_df.to_csv(OUTPUTS["danger_csv"], index=False)

# Plot 1: AP vs coverage of the FIXED HARD_FP coordinates that lie in truly-empty R32 regions.
fig, ax = plt.subplots(figsize=(10, 6))
for recall_target in ACTIVE_REGION_RECALL_GRID:
    frame = frontier_summary[
        frontier_summary["active_region_recall_target"] == float(recall_target)
    ].sort_values("fixed_hard_fp_empty_coordinate_coverage")
    ax.plot(
        frame["fixed_hard_fp_empty_coordinate_coverage"],
        frame["median_ap1"],
        marker="o",
        markersize=3,
        label=f"active recall={recall_target:.3f}",
    )
ax.axhline(TARGET_AP, linestyle="--", label="target AP1=0.20")
ax.axhline(EXPECTED_BASELINE_AP, linestyle=":", label="EXP045 baseline")
ax.set_xlabel("Coverage of fixed HARD_FP coordinates in truly-empty R32 regions")
ax.set_ylabel("Median natural nucleotide AP1 across fixed FN seeds")
ax.set_title("EXP046 R32 feasibility frontier")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUTS["plot_lines"], dpi=170)
plt.close(fig)

# Plot 2: median AP surface. X is the fraction of all truly-empty regions removed
# from the top of the fixed danger ranking; HARD_FP coverage remains in the CSV.
pivot = frontier_summary.pivot(
    index="active_region_recall_target",
    columns="empty_region_suppression_fraction_target",
    values="median_ap1",
).sort_index(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot.to_numpy(), aspect="auto", origin="upper")
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_yticklabels([f"{x:.3f}" for x in pivot.index])
ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_xticklabels([f"{x:g}" for x in pivot.columns], rotation=60, ha="right")
ax.set_ylabel("Active-region recall target")
ax.set_xlabel("Danger-ranked truly-empty R32 suppression fraction")
ax.set_title("Median natural nucleotide AP1 — R32 feasibility frontier")
fig.colorbar(im, ax=ax, label="AP1")
fig.tight_layout()
fig.savefig(OUTPUTS["plot_heatmap"], dpi=170)
plt.close(fig)

summary = {
    "experiment_id": EXPERIMENT_ID,
    "stage": "R32 feasibility frontier; validation-label diagnostic only",
    "region_size": REGION_SIZE,
    "baseline_ap1_reproduced": float(baseline_ap["overall"]),
    "oracle32_ap1_reproduced": float(oracle32_ap),
    "target_ap1": TARGET_AP,
    "positions": EXPECTED_TOTAL_POSITION_STRAND,
    "positives": EXPECTED_POSITIVES,
    "region_stats": region_stats,
    "frontier_definition": {
        "empty_region_order": "truly-empty regions sorted by max EXP045 5dp score descending; region-key tie break",
        "active_false_negatives": "uniform random active-region drops with fixed seeds",
        "seeds": list(FN_SEEDS),
        "suppression": "all base scores in a suppressed region are moved to score bin 0",
        "interpretation": (
            "Sensitivity frontier under an explicit error model. For recall<1 it is not a universal "
            "necessary/sufficient boundary because which active regions are missed matters."
        ),
    },
    "crossings": crossings.where(pd.notna(crossings), None).to_dict(orient="records"),
    "guardrails": {
        "R32_remains_preregistered_primary": True,
        "R16_not_selected_from_validation": True,
        "validation_used_for_training": False,
        "validation_used_for_hard_negative_mining": False,
        "alpha_tuned_on_validation": False,
        "frozen_pilot_not_run_in_this_notebook": True,
    },
    "next_step": (
        "Use this frontier only to quantify feasibility demands. If proceeding, train the frozen regional head "
        "and choose alpha/calibration/hard-negative fraction/weight decay exclusively by train-contig CV."
    ),
    "agents_sha256": AGENTS_SHA256,
    "outputs": {name: str(path) for name, path in OUTPUTS.items()},
}
OUTPUTS["summary_json"].write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Saved frontier outputs to:", FRONTIER_DIR)


{
  "experiment_id": "CNN-EXP-046",
  "stage": "R32 feasibility frontier; validation-label diagnostic only",
  "region_size": 32,
  "baseline_ap1_reproduced": 0.09763009437871803,
  "oracle32_ap1_reproduced": 0.23831825797792802,
  "target_ap1": 0.2,
  "positions": 83807486,
  "positives": 56205,
  "region_stats": {
    "valid_regions": 2675614,
    "active_regions": 28303,
    "empty_regions": 2647311,
    "active_region_prevalence": 0.010578132720190581,
    "fixed_hard_fp_coordinates": 20000,
    "fixed_hard_fp_coordinates_in_empty_r32": 15842,
    "fixed_hard_fp_empty_fraction": 0.7921,
    "unique_empty_r32_regions_containing_fixed_hard_fp": 12986,
    "fixed_low_tp_coordinates": 14052
  },
  "frontier_definition": {
    "empty_region_order": "truly-empty regions sorted by max EXP045 5dp score descending; region-key tie break",
    "active_false_negatives": "uniform random active-region drops with fixed seeds",
    "seeds": [
      4601,
      4602,
      4603,
      4604,
      4

## Interpretation rule after execution

Read the frontier in this order:

1. **Do not change R.** Oracle32 = 0.2383 remains the pre-registered result.
2. Start with the deterministic row active_region_recall = 1.0 to see how much danger-ranked empty-region suppression alone is required for AP1 = 0.20.
3. Then inspect 0.999 / 0.995 / 0.99 / 0.98 / 0.975 / 0.95 recall. Use both the **median** and **min-across-fixed-seeds** crossings.
4. Translate suppression fraction into **fixed HARD_FP-empty coordinate coverage**. This is the most directly relevant measure of how much of the dangerous EXP045 background a learned head would need to remove.
5. Check realized **positive-position recall** and **LOW_TP retention**. Region recall alone is not enough.
6. Do not conclude that a small learned head can achieve a frontier point merely because the diagnostic surface contains it. That is what the train-contig frozen-head CV must test next.

The frozen-head pilot belongs in a **separate notebook/run stage** so that this validation-only diagnostic does not leak into training or hyperparameter selection.


In [11]:
# ------------------------------------------------------------
# CLEAN UP TEMPORARY ~scalar-only CACHE AFTER ALL OUTPUTS EXIST
# ------------------------------------------------------------
for name, path in OUTPUTS.items():
    assert path.is_file(), f"Expected output missing before scratch cleanup: {path}"

# Close Windows memmap handles before deleting scratch files.
for mm in (score_mm, valid_mask_mm, positive_mask_mm, region_key_mm, risk_mm):
    mm.flush()

del score_mm, valid_mask_mm, positive_mask_mm, region_key_mm, risk_mm
gc.collect()
shutil.rmtree(SCRATCH_DIR)
assert not SCRATCH_DIR.exists()
print("Temporary scalar region cache removed.")
print("EXP046 R32 feasibility frontier complete.")


NameError: name 'score_mm' is not defined